# Reproducing an Argo file with tabray

Argo float 1902500, synthetic profile file (`Sprof`): 20 profiles x 520 levels.

- sections 1-4: what the file holds, read with `DatasetDescription.describe_dataset`
- sections 5-8: generate a tabray dataset in netCDF and parquet with the same grid, occupied sites
  and distinct values per variable, QC flags and strings included
- section 9: sizes of the synthetic dataset against the original file

The synthetic values are integer codes (strings: codes written as text): the sizes depend on how
many distinct values a column holds, not on what they are.

In [1]:
import contextlib
import io
import timeit
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from tabray.generate_data import GenerateData
from tabray.output import DatasetDescription
from tabray.output.variable_encoding import VariableEncoding
from tabray.utils.viz import format_bytes

NC_PATH = Path("/home/enrico/myWHOI/playground/tabray/argo/1902500_Sprof.nc")
OUT_DIR = Path("argo_reproduced")

## 1. Open the file

In [2]:
ds_argo = xr.open_dataset(NC_PATH)

## 2. Description, one variable at a time

In [3]:
df_argo_description = DatasetDescription.describe_dataset(str(NC_PATH))

print("grid  netCDF: ", df_argo_description.attrs["grid"])

with pd.option_context("display.max_rows", 200):
    display(df_argo_description)

grid  netCDF:  {'N_PROF': 20, 'N_PARAM': 10, 'N_CALIB': 1, 'N_LEVELS': 520}


,dims,var_sites,var_sites_occupied,density,overlap,overlap_reverse,unused_coords,dtype
PRES,"(N_PROF, N_LEVELS)",10400,10252,0.985769,NaN,NaN,0,float32
DATA_TYPE,(),1,1,1.000000,NaN,NaN,0,|S1
FORMAT_VERSION,(),1,1,1.000000,NaN,NaN,0,|S1
HANDBOOK_VERSION,(),1,1,1.000000,NaN,NaN,0,|S1
REFERENCE_DATE_TIME,(),1,1,1.000000,NaN,NaN,0,|S1
DATE_CREATION,(),1,1,1.000000,NaN,NaN,0,|S1
DATE_UPDATE,(),1,1,1.000000,NaN,NaN,0,|S1
PLATFORM_NUMBER,"(N_PROF,)",20,20,1.000000,1.000000,1.0,0,|S1
PROJECT_NAME,"(N_PROF,)",20,20,1.000000,1.000000,1.0,0,|S1
PI_NAME,"(N_PROF,)",20,20,1.000000,1.000000,1.0,0,|S1


## 3. Selecting Variables tabray can reproduce

tabray can reproduce most (but not all) the variables in the Argo Sprof file. We'll have to leave out scalars (like `DATA_TYPE`) and variables with coordinates `N_PARAM` and/or `N_CALIB`. The latter is because tabray assumes that at least one variable (`var0`) exists on all dimensions, but this is not true for Argo's profile file. We'll then assume that `PRES` is tabray's `var0`, and drop variables on dimensions different than `N_PROF` and `N_LEVELS`: these are very small and we'll show where leaving out ~1-2% of the data.

Variables reproduced:
- profile variables: float, on `(N_PROF, N_LEVELS)`, holding at least one value (32)
- per-profile variables: numeric or time, on `(N_PROF,)` (6)
- QC flags on either dims (31): one character each (`|S1`, 1 byte) with 1-3 distinct values,
  generated as `int8`, also 1 byte
- strings on `(N_PROF,)` (10): `S<k>`, k characters each (`PLATFORM_NUMBER` is `S8`), one
  distinct value each
- empty variables (all fill): density 0, written empty

Per variable: occupied sites, density on its own grid, distinct values.

In [4]:
# variables on (N_PROF, N_LEVELS) or (N_PROF,); an empty one gets density 0
is_reproducible = df_argo_description["dims"].isin([("N_PROF", "N_LEVELS"), ("N_PROF",)])
df_argo_vars = df_argo_description.loc[is_reproducible, ["dims", "var_sites_occupied", "density"]]


def tabray_dtype(name: str) -> str:
    """QC flag -> int8 (1 byte); other string -> S<k>; number -> int32."""
    if name.endswith("_QC"):
        return "int8"
    if df_argo_description.loc[name, "dtype"].startswith("|S"):
        # k = width of the padded strings in the file
        return f"S{max(len(text) for text in ds_argo[name].values.ravel() if isinstance(text, bytes))}"
    return "int32"


df_argo_vars["dtype"] = [tabray_dtype(name) for name in df_argo_vars.index]
df_argo_vars["distinct"] = [
    int(pd.Series(ds_argo[name].values.ravel()).nunique()) for name in df_argo_vars.index
]
# var0 must hold the most sites: sort, stable so equal counts keep file order
df_argo_vars = df_argo_vars.sort_values("var_sites_occupied", ascending=False, kind="stable")
n_prof, n_levels = ds_argo.sizes["N_PROF"], ds_argo.sizes["N_LEVELS"]
with pd.option_context("display.max_rows", 100):
    display(df_argo_vars)

,dims,var_sites_occupied,density,dtype,distinct
PRES,"(N_PROF, N_LEVELS)",10252,0.985769,int32,942
PRES_QC,"(N_PROF, N_LEVELS)",10252,0.985769,int8,1
PRES_ADJUSTED,"(N_PROF, N_LEVELS)",10252,0.985769,int32,7387
PRES_ADJUSTED_QC,"(N_PROF, N_LEVELS)",10252,0.985769,int8,1
TEMP,"(N_PROF, N_LEVELS)",10252,0.985769,int32,7022
TEMP_QC,"(N_PROF, N_LEVELS)",10252,0.985769,int8,2
TEMP_dPRES,"(N_PROF, N_LEVELS)",10252,0.985769,int32,36
TEMP_ADJUSTED,"(N_PROF, N_LEVELS)",10252,0.985769,int32,7022
TEMP_ADJUSTED_QC,"(N_PROF, N_LEVELS)",10252,0.985769,int8,2
PSAL,"(N_PROF, N_LEVELS)",10252,0.985769,int32,1733


## 4. Determining variables ranges

In tabray, a variable with `value_range=(0, N-1)` can take `N` distinct values (or 'choices'), and draws each value uniformly among them.
In general, given `m` draws,  the expected number of distinct values drawn from `value_range` is `N * (1 - exp(-m / N))`.
`choices_for()` picks `N` so that this equals the distinct count measured from Argo Sprof file (`distinct` column in previous cell), capped by what the dtype can express
(`int8`: 127 values, `S<k>`: `36**k` values). One distinct value gives `N = 1`: a constant.

In [5]:
def choices_for(distinct: int, draws: int, capacity: int) -> int:
    """Number of choices N whose `draws` uniform picks give `distinct` values on average.

    - expected distinct = N * (1 - exp(-draws / N)), rising with N towards `draws`
    - bisection on N, then the closer of the two bracketing N
    - at most `capacity`, the codes the dtype holds; `distinct == draws` takes all of them
    - `distinct <= 1`: N = 1, a constant (or an empty variable)
    """

    # balls into bins: expected non-empty bins (Mitzenmacher & Upfal, Probability and Computing, ch. 5)
    def expected(n_choices):
        return n_choices * -np.expm1(-draws / n_choices)

    if distinct <= 1:
        return 1
    if distinct >= draws or expected(capacity) <= distinct:
        return capacity
    low, high = distinct, capacity
    while high - low > 1:
        mid = (low + high) // 2
        if expected(mid) < distinct:
            low = mid
        else:
            high = mid
    closer = low if distinct - expected(low) < expected(high) - distinct else high
    return closer


def capacity_of(dtype: str) -> int:
    """Codes a dtype holds: int8 0..126 (fill -127), S<k> 36**k, int32 2**30.

    Capped at 2**30: wider strings hold more codes than a float can count.
    """
    if dtype == "int8":
        return 127
    if dtype.startswith("S"):
        return min(VariableEncoding.TEXT_BASE ** int(dtype[1:]), 2**30)
    return 2**30


df_argo_vars["choices"] = [
    choices_for(row.distinct, row.var_sites_occupied, capacity_of(row.dtype))
    for row in df_argo_vars.itertuples()
]

## 5. Build Argo-like parameters and generate synthetic datasets

- grid: `num_obs` = sites of the most occupied variable, `ratio_dims=(1, 26)` -> 20 x 520
- `layout="padded"` along levels: each profile fills its first k levels, as Argo does
- per-profile variables: `var_dims=[0]`; density capped at var0's (0.986 instead of 1.0),
  which still occupies all 20 profiles
- empty variables: density 0
- `dtype`: `int8` for QC flags, `S<k>` for other strings, `int32` otherwise
- `value_range=(0, N-1)`; one distinct value gives `(0, 0)`, a constant

In [6]:
num_obs_argo = int(df_argo_vars["var_sites_occupied"].iloc[0])
num_vars_argo = len(df_argo_vars)
ratio_dims_argo = (1, n_levels / n_prof)  # exact ratio: integer division changes the grid
# validate_density_refvar: no variable may exceed var0's density
var0_density = df_argo_vars["density"].iloc[0]
density_argo = list(df_argo_vars["density"].clip(upper=var0_density)) #tabray does not allow vars to have density larger than var0's
var_dims_argo = [2 if len(dims) == 2 else [0] for dims in df_argo_vars["dims"]]
dtype_argo = list(df_argo_vars["dtype"])
value_range_argo = [(0, n_choices - 1) for n_choices in df_argo_vars["choices"]]

with contextlib.redirect_stdout(io.StringIO()):
    gen = GenerateData(
        num_obs=num_obs_argo,
        num_dims=2,
        ratio_dims=ratio_dims_argo,
        num_vars=num_vars_argo,
        density=density_argo,
        var_dims=var_dims_argo,
        layout="padded",
        padded_dim=1,
        dtype=dtype_argo,
        value_range=value_range_argo,
        seed=1,
    )
    gen.generate(
        netcdf_filepath=str(OUT_DIR / "synthetic/nc/argo.nc"),
        parquet_filepath=str(OUT_DIR / "synthetic/pq/argo.parquet"),
    )
SYN_NC = OUT_DIR / "synthetic/nc/argo.nc"
SYN_PQ = next((OUT_DIR / "synthetic/pq").glob("*.parquet"))
ds_syn = xr.open_dataset(SYN_NC)
print("grid:", dict(ds_syn.sizes))

grid: {'x0': 20, 'x1': 520}


## 6. Synthetic vs reference, per variable

In [7]:
var_names = {f"var{i}": name for i, name in enumerate(df_argo_vars.index)}
df_syn = pd.read_parquet(SYN_PQ)
df_syn_description = DatasetDescription.describe_dataset(str(SYN_NC)).rename(index=var_names)
df_comparison = pd.DataFrame(
    {
        "occupied, real": df_argo_vars["var_sites_occupied"],
        "occupied, synthetic": df_syn_description["var_sites_occupied"],
        "distinct, real": df_argo_vars["distinct"],
        "distinct, synthetic": df_syn[list(var_names)].nunique().rename(var_names),
    }
)
with pd.option_context("display.max_rows", 100):
    display(df_comparison)

,"occupied, real","occupied, synthetic","distinct, real","distinct, synthetic"
PRES,10252,10252,942,942
PRES_QC,10252,10252,1,1
PRES_ADJUSTED,10252,10252,7387,7375
PRES_ADJUSTED_QC,10252,10252,1,1
TEMP,10252,10252,7022,7030
TEMP_QC,10252,10252,2,2
TEMP_dPRES,10252,10252,36,36
TEMP_ADJUSTED,10252,10252,7022,7038
TEMP_ADJUSTED_QC,10252,10252,2,2
PSAL,10252,10252,1733,1732


## 7. Sizes: synthetic vs original

The original file holds more variables than tabray reproduces, and is compressed (zlib 4 with
shuffle); tabray writes netCDF uncompressed. To compare like with like, the original file
restricted to the variables of section 5 and the synthetic netCDF are each written twice, with
the same encoding: uncompressed, and zlib 4 with shuffle. Memory counts values in their stored
dtype.

No parquet version of the original exists, so the synthetic parquet is reported on its own,
against the synthetic netCDF.

In [8]:
COMPARE_DIR = OUT_DIR / "compare"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)
ZLIB = {"zlib": True, "complevel": 4, "shuffle": True}
# stored dtypes, no decoding: the bytes each file holds
ds_argo_stored = xr.open_dataset(NC_PATH, decode_cf=False)[list(df_argo_vars.index)]
ds_syn_stored = xr.open_dataset(SYN_NC, decode_cf=False)


def netcdf_sizes(ds: xr.Dataset, name: str) -> dict:
    """In-memory size, and on-disk size uncompressed and with ZLIB."""
    sizes = {"in memory": int(ds.load().nbytes)}
    for label, encoding in (("netCDF uncompressed", {"zlib": False}), ("netCDF zlib 4 + shuffle", ZLIB)):
        path = COMPARE_DIR / f"{name}_{label.split()[1]}.nc"
        ds.to_netcdf(path, encoding={var: encoding for var in ds.data_vars})
        sizes[label] = path.stat().st_size
    return sizes


size_argo, size_syn = netcdf_sizes(ds_argo_stored, "argo"), netcdf_sizes(ds_syn_stored, "syn")
df_sizes = pd.DataFrame({"original": size_argo, "synthetic": size_syn})
df_sizes["synthetic vs original"] = [
    f"{100 * (size_syn[key] - size_argo[key]) / size_argo[key]:+.1f} %" for key in size_argo
]
df_sizes[["original", "synthetic"]] = df_sizes[["original", "synthetic"]].map(format_bytes)
display(df_sizes)

df_syn_sizes = pd.Series(
    {
        "parquet on disk": format_bytes(SYN_PQ.stat().st_size),
        "parquet rows": len(df_syn),
        "pandas in memory": format_bytes(df_syn.memory_usage(deep=True).sum()),
        "parquet / netCDF uncompressed": round(SYN_PQ.stat().st_size / size_syn["netCDF uncompressed"], 2),
        "parquet / netCDF zlib 4 + shuffle": round(SYN_PQ.stat().st_size / size_syn["netCDF zlib 4 + shuffle"], 2),
    },
    name="synthetic",
    dtype=object,  # sizes, a row count and ratios side by side
)
df_syn_sizes

/home/enrico/miniforge3/envs/tabray/lib/python3.14/site-packages/xarray/namedarray/core.py:261: UserWarning: Duplicate dimension names present: dimensions {'string1'} appear more than once in dims=('x0', 'string1', 'string1'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  self._dims = self._parse_dimensions(dims)


,original,synthetic,synthetic vs original
in memory,1.75 MiB,1.76 MiB,+0.2 %
netCDF uncompressed,1.87 MiB,1.83 MiB,-2.3 %
netCDF zlib 4 + shuffle,801.75 KiB,786.11 KiB,-2.0 %


parquet on disk                      930.74 KiB
parquet rows                              10259
pandas in memory                       6.26 MiB
parquet / netCDF uncompressed               0.5
parquet / netCDF zlib 4 + shuffle          1.18
Name: synthetic, dtype: object

## 8. Timed read: PRES < 200 dbar, per-profile CHLA and NITRATE

- operation: read PRES, CHLA and NITRATE where PRES < 200 dbar; per profile (`x0`), min, max and
  mean of CHLA and NITRATE
- synthetic PRES holds codes, not dbar: the threshold is the code below which the same share of
  values falls as real PRES below 200 dbar
- netCDF: the synthetic netCDF rewritten as `NETCDF4_CLASSIC`, zlib 4 with shuffle, as the original file
- read: netCDF `xr.open_dataset`, load the three variables, mask, keep profiles with a level
  below the threshold; parquet `pd.read_parquet` with `filters` (rows with PRES at or above the
  threshold are dropped on read)
- min, max, mean, each timed on its own from the data read: netCDF reduces along levels (`x1`),
  parquet runs `groupby("x0")`
- timing: median of `N_RUNS` runs; after the first run the file is in the OS cache


In [9]:
syn_names = {name: var for var, name in var_names.items()}
pres, chla, nitrate = syn_names["PRES"], syn_names["CHLA"], syn_names["NITRATE"]
# share of real PRES values below 200 dbar -> synthetic code at the same quantile
pres_share_argo = float((ds_argo["PRES"] < 200).sum() / ds_argo["PRES"].notnull().sum())
pres_threshold_syn = float(ds_syn[pres].quantile(pres_share_argo))
STATS = ["min", "max", "mean"]
N_RUNS = 20


def rewrite_like_argo(nc_path: Path) -> Path:
    """Copy of nc_path in the original file's format: NETCDF4_CLASSIC, zlib 4 with shuffle."""
    z4_path = nc_path.with_name(f"{nc_path.stem}_z4.nc")
    with xr.open_dataset(nc_path) as ds:
        for var in ds.data_vars:
            # keep each variable's stored dtype and fill value; chunked, as zlib requires
            ds[var].encoding.update(ZLIB, contiguous=False)
        ds.to_netcdf(z4_path, format="NETCDF4_CLASSIC")
    return z4_path


def read_netcdf(nc_path: Path) -> xr.Dataset:
    """CHLA and NITRATE where PRES < threshold; profiles with no such level dropped."""
    with xr.open_dataset(nc_path) as ds:
        ds_shallow = ds[[pres, chla, nitrate]].load()
    is_shallow = ds_shallow[pres] < pres_threshold_syn  # NaN PRES -> False
    ds_shallow = ds_shallow[[chla, nitrate]].where(is_shallow)
    return ds_shallow.isel(x0=is_shallow.any("x1").values)


def read_parquet(pq_path: Path) -> pd.DataFrame:
    """Rows where PRES < threshold, columns x0, CHLA, NITRATE."""
    return pd.read_parquet(
        pq_path, columns=["x0", chla, nitrate], filters=[(pres, "<", pres_threshold_syn)]
    )


def reduce_netcdf(ds: xr.Dataset, stat: str) -> pd.DataFrame:
    """Per-profile stat of CHLA and NITRATE: reduce along levels."""
    return getattr(ds, stat)("x1").to_dataframe()


def reduce_parquet(df: pd.DataFrame, stat: str) -> pd.DataFrame:
    """Per-profile stat of CHLA and NITRATE: group rows by profile."""
    return df.groupby("x0")[[chla, nitrate]].agg(stat)


def median_seconds(operation) -> float:
    """Median wall time of N_RUNS calls."""
    return float(np.median(timeit.repeat(operation, number=1, repeat=N_RUNS)))


def parquet_bytes(pq_path: Path) -> int:
    """Size of a parquet file, or of the parquet files in a directory."""
    if pq_path.is_dir():
        return sum(part.stat().st_size for part in pq_path.glob("*.parquet"))
    return pq_path.stat().st_size


def time_operations(nc_path: Path, pq_path: Path) -> pd.DataFrame:
    """Median seconds of the read and of each stat, netCDF vs parquet."""
    ds_shallow, df_shallow = read_netcdf(nc_path), read_parquet(pq_path)
    timings = {
        "netCDF": {"read": median_seconds(lambda: read_netcdf(nc_path))},
        "parquet": {"read": median_seconds(lambda: read_parquet(pq_path))},
    }
    for stat in STATS:
        # same numbers from both formats (parquet columns are nullable Int32/Float64)
        pd.testing.assert_frame_equal(
            reduce_netcdf(ds_shallow, stat),
            reduce_parquet(df_shallow, stat).astype(float),
            check_names=False,
        )
        timings["netCDF"][stat] = median_seconds(lambda: reduce_netcdf(ds_shallow, stat))
        timings["parquet"][stat] = median_seconds(lambda: reduce_parquet(df_shallow, stat))
    df_timings = 1000 * pd.DataFrame(timings)  # ms
    df_timings.loc["total"] = df_timings.sum()
    df_timings["netCDF / parquet"] = df_timings["netCDF"] / df_timings["parquet"]
    print("profiles:", ds_shallow.sizes["x0"], " parquet rows read:", len(df_shallow))
    print("netCDF:", format_bytes(nc_path.stat().st_size), " parquet:", format_bytes(parquet_bytes(pq_path)))
    return df_timings.round({"netCDF": 3, "parquet": 3, "netCDF / parquet": 2}).rename_axis("ms, median")


SYN_NC3 = rewrite_like_argo(SYN_NC)
print(f"PRES threshold: code {pres_threshold_syn:.0f} ({pres_share_argo:.1%} of values below)")
df_syn_timings = time_operations(SYN_NC3, SYN_PQ)
df_syn_timings

PRES threshold: code 185 (19.7% of values below)
profiles: 20  parquet rows read: 2014
netCDF: 783.51 KiB  parquet: 930.74 KiB


,netCDF,parquet,netCDF / parquet
"ms, median",,,
read,12.671,2.724,4.65
min,0.388,0.610,0.63
max,0.377,0.614,0.61
mean,0.450,0.629,0.72
total,13.886,4.577,3.03


## 9. More data

The same float recording more profiles, or deeper profiles. Per case, profiles and levels are
scaled; `num_obs` scales with their product, `ratio_dims` with their ratio:

- x10 profiles: 200 profiles x 520 levels
- x10 levels: 20 profiles x 5200 levels

Densities, dims, layout, dtypes and value ranges as in section 5: the same number of choices per
variable, drawn more often. A case with more than `MAX_OBS` sites is generated in parallel
chunks of at most `MAX_OBS` (about 1 GB of memory each), `MAX_WORKERS` at a time.


In [12]:
CASES = {"x10 profiles": (10, 1), "x10 levels": (1, 20)}  # (profiles, levels)
MAX_OBS = 10 * num_obs_argo
MAX_WORKERS = 4


def generate_scaled(prof_scale: int, level_scale: int) -> tuple[Path, Path]:
    """Generate with profiles and levels scaled; netCDF path and parquet directory."""
    out_dir = OUT_DIR / f"synthetic_p{prof_scale}_l{level_scale}"
    nc_path = out_dir / "nc/argo.nc"
    with contextlib.redirect_stdout(io.StringIO()):
        gen = GenerateData(
            num_obs=prof_scale * level_scale * num_obs_argo,
            num_dims=2,
            ratio_dims=(1, level_scale * n_levels / (prof_scale * n_prof)),
            num_vars=num_vars_argo,
            density=density_argo,
            var_dims=var_dims_argo,
            layout="padded",
            padded_dim=1,
            dtype=dtype_argo,
            value_range=value_range_argo,
            seed=1,
            max_obs=MAX_OBS,
            max_workers=MAX_WORKERS,
        )
        gen.generate(
            netcdf_filepath=str(nc_path),
            parquet_filepath=str(out_dir / "pq/argo.parquet"),
            merge_nc=True,  # parallel chunks -> one netCDF
        )
    return nc_path, out_dir / "pq"


df_scaled_timings = {}
for case, (prof_scale, level_scale) in CASES.items():
    nc_path, pq_path = generate_scaled(prof_scale, level_scale)
    with xr.open_dataset(nc_path) as ds:
        print(f"{case}, grid:", dict(ds.sizes))
    df_scaled_timings[case] = time_operations(rewrite_like_argo(nc_path), pq_path)
    display(df_scaled_timings[case])

x10 profiles, grid: {'x0': 200, 'x1': 520}
profiles: 200  parquet rows read: 20262
netCDF: 4.75 MiB  parquet: 7.56 MiB


,netCDF,parquet,netCDF / parquet
"ms, median",,,
read,21.733,7.305,2.98
min,0.474,0.859,0.55
max,0.466,0.849,0.55
mean,0.884,0.922,0.96
total,23.557,9.934,2.37


Saved to argo_reproduced/synthetic_p1_l20/nc/argo_0.nc
Saved to argo_reproduced/synthetic_p1_l20/pq/tmp_argo/chunk_0000_*.parquet
Saved to argo_reproduced/synthetic_p1_l20/nc/argo_1.nc
Saved to argo_reproduced/synthetic_p1_l20/pq/tmp_argo/chunk_0001_*.parquet
x10 levels, grid: {'x0': 20, 'x1': 10400}
profiles: 20  parquet rows read: 40437
netCDF: 9.14 MiB  parquet: 14.31 MiB


,netCDF,parquet,netCDF / parquet
"ms, median",,,
read,23.367,7.966,2.93
min,0.419,1.013,0.41
max,0.418,1.014,0.41
mean,1.281,1.199,1.07
total,25.484,11.192,2.28


In [11]:
ds_argo.close()
ds_syn.close()
ds_argo_stored.close()
ds_syn_stored.close()